We're going to illustrate some practical differences between CPU and GPU operations using Python and PyTorch. First, we'll import some modules.

In [ ]:
import torch
import time
import statistics


The notebook chooses CUDA, then Apple MPS, and otherwise CPU. In Colab, select a GPU runtime if you want a GPU comparison. CPU-only runs remain useful for comparing implementations.


In [ ]:
def get_best_device():
    # CUDA includes ROCm builds, which also expose torch.cuda.
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = get_best_device()
print(f"Using device: {device}")


We measure computation on preallocated inputs. Random generation and CPU-to-GPU transfers happen before timing. Each implementation gets warm-up runs, then seven synchronized samples; we report the median and interquartile range (IQR). The IQR shows measurement variability rather than hiding it behind a single number.

Synchronization before and after each GPU sample includes completion of the work. `perf_counter()` supplies a monotonic clock. These are end-to-end operation timings, including Python dispatch and synchronization overhead; they are not kernel-only timings.


In [ ]:
def synchronize(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def prepare_inputs(size, accelerator):
    generator = torch.Generator().manual_seed(2025 + size)
    cpu = (torch.randn(size, generator=generator), torch.randn(size, generator=generator))
    inputs = {"cpu": cpu}
    if accelerator.type != "cpu":
        inputs["gpu"] = tuple(tensor.to(accelerator) for tensor in cpu)
        synchronize(accelerator)
    return inputs


def calculate_and_time(func, inputs, warmup=2, repeats=7):
    if warmup < 1 or repeats < 4:
        raise ValueError("Use at least one warm-up and four timing samples.")
    times = {"gpu_type": None, "gpu_iqr": None}
    times["gpu"] = None
    for label, (a, b) in inputs.items():
        samples = []
        with torch.inference_mode():
            # Verify the implementation against the same input, outside timing.
            torch.testing.assert_close(func(a, b), torch.dot(a, b), rtol=1e-3, atol=1e-3)
            for _ in range(warmup):
                func(a, b)
            for _ in range(repeats):
                synchronize(a.device)
                start = time.perf_counter()
                result = func(a, b)
                synchronize(a.device)
                samples.append(time.perf_counter() - start)
                del result
        q1, _, q3 = statistics.quantiles(samples, n=4)
        times[label] = statistics.median(samples)
        times[label + "_iqr"] = q3 - q1
        if label == "gpu":
            times["gpu_type"] = a.device.type
    return times


We'll be performing dot products on vectors of various sizes. Specifically, we'll be computing the dot product between two vectors `A` and `B` both with shape `Nx1`. `N` will be one of `[10, 50, 100, 500, 1000, 2000, 5000]`.

In [ ]:
# Varying data sizes
data_sizes = [10, 50, 100, 500, 1000, 2000, 5000]

Compare a deliberately slow element-by-element loop, a vectorized multiply-and-sum, and `torch.dot`. All three use the same vectors for a given size and perform O(N) arithmetic. The vectorized version creates an intermediate product tensor; the loop dispatches many small operations, especially costly on a GPU. Keep the loop to observe that cost rather than treating it as a production implementation.


In [ ]:
def dot_product_for_loop(a, b):
    result = a.new_zeros(())
    for i in range(a.shape[0]):
        result += a[i] * b[i]
    return result


def dot_product_vectorized(a, b):
    return (a * b).sum()


def dot_product_torch(a, b):
    return torch.dot(a, b)


Prepare inputs once per size and reuse them across all implementations. Only one size remains allocated at a time; the results table holds timings, not tensors.


In [ ]:
results = []
implementations = {
    "For loop": dot_product_for_loop,
    "Vectorized": dot_product_vectorized,
    "Torch dot": dot_product_torch,
}
for size in data_sizes:
    inputs = prepare_inputs(size, device)
    for name, function in implementations.items():
        results.append({"name": name, "size": size, **calculate_and_time(function, inputs)})
    del inputs


Report median and IQR in microseconds. `N/A` means this run had no GPU; it is not a zero-time measurement.


In [ ]:
print("Implementation | N | CPU median / IQR (us) | GPU median / IQR (us)")
for row in results:
    cpu = f"{row['cpu'] * 1e6:.2f} / {row['cpu_iqr'] * 1e6:.2f}"
    gpu = "N/A" if row["gpu"] is None else f"{row['gpu'] * 1e6:.2f} / {row['gpu_iqr'] * 1e6:.2f}"
    print(f"{row['name']:12} | {row['size']:5} | {cpu:24} | {gpu}")


### Interpret your measurements

- Compare implementations on the same device and size. The scalar loop exposes dispatch overhead; vectorization reduces dispatches while still doing O(N) arithmetic.
- Inspect the IQR. When differences are comparable to the variability, repeat the experiment before declaring a winner.
- A GPU may remain slower for every size here, including 5000. Identify a crossover only if your measured results support one. For larger vectorized experiments, expand sizes within memory limits and omit the scalar GPU loop if it becomes impractical.
- Allocation and transfers are excluded. These results cannot establish whether moving a CPU workload to the GPU is worthwhile end to end. Measure transfer and allocation costs separately for that question.
- Device initialization and first-use setup are warmed up. Cold-start applications have a different cost profile.
- Record PyTorch version, device, CPU thread count, sizes, and repeat count with your conclusions. Do not assume another computer will reproduce your timings.

For further experiments, see the [PyTorch benchmark guide](https://docs.pytorch.org/tutorials/recipes/recipes/benchmark.html).
